In [1]:
import hashlib
import numpy as np

def sha_bit_mirror(twin_mid=4.0, header=b'314', rounds=8):
    # Encode twin_mid first for corrected input stream
    input_data = str(twin_mid).encode() + header
    sha = hashlib.sha256(input_data)
    digest = sha.digest()
    # Bit mirror logic via XOR roll
    bits = np.unpackbits(np.frombuffer(digest, dtype=np.uint8))
    mirrored = bits ^ np.roll(bits, 1)
    h = np.sum(mirrored) / len(mirrored)  # H-ratio
    return h, digest.hex()

# Midpoint test injection
mids = [4.0, 6.0, 12.0]
h_vals = []

for mid in mids:
    h, digest = sha_bit_mirror(twin_mid=mid)
    h_vals.append(h)
    print(f"Midpoint: {mid}, H: {h:.3f}")
    if abs(h - 0.35) < 0.05:
        print(f"Resonance Lock Achieved at mid={mid}")
        break

print("H-vals:", h_vals)


Midpoint: 4.0, H: 0.492
Midpoint: 6.0, H: 0.562
Midpoint: 12.0, H: 0.539
H-vals: [0.4921875, 0.5625, 0.5390625]


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.integrate import quad
from scipy.signal import find_peaks
from numpy import sin, cos, pi, sqrt, exp

# Constants
H = 0.35  # Harmonic constant
k = 1     # Layer index
num_glyphs = 500  # Number of glyphs to simulate
l = 5     # OAM-like mode index for probe

# Generate spiral coordinates using modified Sacks mapping
n = np.arange(1, num_glyphs + 1)
r = np.sqrt(n)
theta = 2 * pi * np.sqrt(n) % (2 * pi)
theta_prime = theta + 2 * pi * H * k

# Define harmonic glyph function
def glyph_function(r_val, theta_val, A_k=[1, 0.5, 0.25], phi_k=[0, pi/4, pi/2]):
    return sum(
        A_k[i] * sin(2 * pi * (i + 1) * r_val * cos(theta_val) + phi_k[i])
        for i in range(len(A_k))
    )

# Compute glyph values
glyph_values = np.array([glyph_function(r[i], theta_prime[i]) for i in range(num_glyphs)])

# Define probe function and compute overlap
probe_function = lambda theta_val: exp(1j * l * theta_val)
overlap_integrals = np.real([glyph_values[i] * np.real(probe_function(theta_prime[i])) for i in range(num_glyphs)])

# Threshold for resonance detection
tau = 0.5
resonant_indices = np.where(overlap_integrals > tau)[0]

# Prepare DataFrame for display
glyph_data = pd.DataFrame({
    'Index': n,
    'r': r,
    'theta': theta_prime,
    'GlyphValue': glyph_values,
    'Overlap': overlap_integrals,
    'Resonant': np.isin(np.arange(num_glyphs), resonant_indices)
})

import ace_tools as tools; tools.display_dataframe_to_user(name="Spiral Glyph Lattice with Resonance Detection", dataframe=glyph_data)

# Plot spiral with resonance highlighted
plt.figure(figsize=(10, 8))
x = r * np.cos(theta_prime)
y = r * np.sin(theta_prime)
sns.scatterplot(x=x, y=y, hue=glyph_data['Resonant'], palette={True: 'red', False: 'gray'}, legend=False)
plt.title("Spiral Glyph Lattice (Red = Resonant Glyphs)")
plt.xlabel("x = r cos(θ')")
plt.ylabel("y = r sin(θ')")
plt.axis('equal')
plt.grid(True)
plt.show()


ModuleNotFoundError: No module named 'ace_tools'